UTS Data Science

*   Nama : Novika Ardiyaningtyas
*   NIM : 250401020135
*   Kelas: IF401


In [1]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset('penguins')

# Pilih kolom yang digunakan
cols = ['species','island','sex',
        'bill_length_mm','bill_depth_mm',
        'flipper_length_mm','body_mass_g']

df = df[cols].copy()

print("Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDistribusi Species:")
print(df['species'].value_counts(normalize=True).round(3))

Shape: (344, 7)

Missing Values:
species               0
island                0
sex                  11
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
dtype: int64

Distribusi Species:
species
Adelie       0.442
Gentoo       0.360
Chinstrap    0.198
Name: proportion, dtype: float64


Pada tahap ini dilakukan pemuatan dan eksplorasi awal dataset Penguins. Hasil pemeriksaan menunjukkan jumlah data, tipe data setiap kolom, serta keberadaan missing values. Analisis awal ini membantu memahami struktur dataset sebelum dilakukan proses pembersihan dan persiapan data.

In [2]:
# Numerik → median
df['bill_length_mm'] = df['bill_length_mm'].fillna(df['bill_length_mm'].median())
df['bill_depth_mm'] = df['bill_depth_mm'].fillna(df['bill_depth_mm'].median())
df['flipper_length_mm'] = df['flipper_length_mm'].fillna(df['flipper_length_mm'].median())
df['body_mass_g'] = df['body_mass_g'].fillna(df['body_mass_g'].median())

# Kategorik → modus
df['sex'] = df['sex'].fillna(df['sex'].mode()[0])
df['island'] = df['island'].fillna(df['island'].mode()[0])

print("Missing setelah handling:")
print(df.isnull().sum())

Missing setelah handling:
species              0
island               0
sex                  0
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
dtype: int64


Pada titik ini, penanganan data yang hilang dilakukan dengan median untuk variabel numerik dan modus untuk variabel kategorik. Setelah langkah-langkah ini selesai, semua nilai yang hilang berhasil diatasi, sehingga dataset menjadi lebih lengkap dan siap untuk dianalisis dan dimodelkan.

In [3]:
print("=== KOLOM SEBELUM ENCODING ===")
print(df.columns.tolist())

df = pd.get_dummies(
    df,
    columns=['species', 'island', 'sex'],
    drop_first=True,
    dtype=int
)

print("\n=== KOLOM SETELAH ENCODING ===")
print(df.columns.tolist())

print("\nContoh Data:")
print(df.head())

=== KOLOM SEBELUM ENCODING ===
['species', 'island', 'sex', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

=== KOLOM SETELAH ENCODING ===
['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'species_Chinstrap', 'species_Gentoo', 'island_Dream', 'island_Torgersen', 'sex_Male']

Contoh Data:
   bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g  \
0           39.10           18.7              181.0       3750.0   
1           39.50           17.4              186.0       3800.0   
2           40.30           18.0              195.0       3250.0   
3           44.45           17.3              197.0       4050.0   
4           36.70           19.3              193.0       3450.0   

   species_Chinstrap  species_Gentoo  island_Dream  island_Torgersen  sex_Male  
0                  0               0             0                 1         1  
1                  0               0             0                 1         0  
2           

Untuk mengubah data kategorik menjadi data numerik, proses encoding dilakukan dengan menggunakan  One-Hot Encoding. Langkah ini diperlukan karena algoritma pembelajaran mesin biasanya hanya dapat memproses data dalam bentuk numerik. Dataset menghasilkan beberapa kolom baru sebagai hasil dari encoding.

In [4]:
from sklearn.model_selection import train_test_split

# Target: species_Gentoo
X = df.drop('species_Gentoo', axis=1)
y = df['species_Gentoo']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape[0]} baris")
print(f"Test : {X_test.shape[0]} baris")

Train: 275 baris
Test : 69 baris


Jumlah data dibagi menjadi data training dan data testing dengan perbandingan 80:20. Tujuan pembagian ini adalah agar model dapat dilatih dengan data pelatihan dan dievaluasi dengan data penilaian. Dengan menggunakan stratify, proporsi kelas tetap seimbang di kedua kelompok data.

In [5]:
from sklearn.preprocessing import StandardScaler

num_cols = [
    'bill_length_mm',
    'bill_depth_mm',
    'flipper_length_mm',
    'body_mass_g'
]

scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

X_test[num_cols] = scaler.transform(X_test[num_cols])

print("Mean scaler:")
print(scaler.mean_.round(2))

print("\nStd scaler:")
print(scaler.scale_.round(2))

print("\nContoh data training:")
print(X_train.head())

print("\nData siap digunakan untuk Machine Learning!")

Mean scaler:
[  43.96   17.17  200.88 4202.55]

Std scaler:
[  5.4    2.02  13.96 784.97]

Contoh data training:
     bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g  \
333        1.397688      -0.429827           2.085571     1.652863   
105       -0.788515       0.858752          -1.208533    -0.831295   
300        0.953036      -1.321920           0.796574     0.538176   
110       -1.084950      -0.330705          -0.205979    -0.480965   
322        0.601021      -0.826313           1.011407     0.984051   

     species_Chinstrap  island_Dream  island_Torgersen  sex_Male  
333                  0             0                 0         1  
105                  0             0                 0         1  
300                  0             0                 0         0  
110                  0             0                 0         0  
322                  0             0                 0         0  

Data siap digunakan untuk Machine Learning!


Untuk memastikan bahwa seluruh fitur memiliki skala yang sama, feature scaling dilakukan pada variabel numerik menggunakan StandardScaler. Proses ini meningkatkan performa dan stabilitas beberapa algoritma pengajaran mesin yang sensitif terhadap perbedaan skala data.

Kesimpulan: Pada praktikum ini dipelajari tahapan dasar preprocessing data untuk Machine Learning, mulai dari eksplorasi data, penanganan missing values, encoding variabel kategorik, pembagian data training dan testing, hingga normalisasi fitur numerik. Hasil akhir menunjukkan bahwa dataset telah bersih, terstruktur, dan siap digunakan untuk proses pelatihan model Machine Learning pada tahap selanjutnya.